# Ovary dataset selection

Nowy narząd, ta sama metodyka co `brain_dataset.ipynb`: `msi_dataset_manager.exploration.DatasetExplorer` + `DatasetReview`/`DatasetReviewProfile` (`packages/msi_dataset_manager/src/msi_dataset_manager/exploration/dataset_review.py`) do obiektywnej detekcji duplikatów, wariantów QC i heurystyki morfologii. Zero zmian w bibliotece, zero pobrań surowych danych — wynikiem jest wyłącznie przejrzana lista kandydatów wyeksportowana jako `filter.json`/`selection.json`.

**Uwaga na wielkość puli — to jest bardzo mało: tylko 5 kandydatów** przy `Mouse`+`Negative` (surowe METASPACE: 297, wszystkie organizmy/polaryzacje). To zdecydowanie NIE będzie duży wkład do korpusu — jeśli liczysz na dużo danych z ovary, ten narząd tego nie da przy obecnym filtrze technologicznym.

**Zakres m/z: `mz_min=200, mz_max=900`, przyjęty jako domyślny punkt startowy spójny z kidney** (patrz `brain_dataset.ipynb`, sekcja 5, gdzie ten wariant wypadł najlepiej na dostępnym katalogu). To NIE jest wynik osobnej optymalizacji dla tego narządu — ujednolicenie zakresu m/z między wszystkimi narządami jest świadomie odłożone na później (tak jak w `kidney_dataset_repaired.ipynb`/`liver_dataset_repaired.ipynb`). Poniżej pokazuję explicité, ile kandydatów ten zakres realnie pokrywa, żebyś widział koszt tego wyboru dla tego konkretnego narządu.

In [1]:
import os
from pathlib import Path

current_path = Path.cwd().resolve()
repository_root = next(
    path
    for path in (current_path, *current_path.parents)
    if (path / "pyproject.toml").is_file()
)
os.chdir(repository_root)

repository_root

PosixPath('/home/max/repositories/MSIAutoEncoderWrapper')

In [2]:
import re

import pandas as pd
from IPython.display import display

from msi_dataset_manager.exploration import DatasetExplorer, DatasetReviewProfile

# REMARK: date i download DB is 12.08.2026 (DD, MM, YYYY) -- same cache as the other notebooks.
explorer = DatasetExplorer(
    source="metaspace",
    cache_dir="assets/local/datasets/metaspace",
    refresh_cache=False,
)

## 1. Szeroka pula kandydatów

Ten sam filtr biologiczny co `brain_dataset.ipynb`: `condition=["Wildtype", "Wtype", "N/A"]`, `organism=Mouse`, `polarity=Negative`. Bez `mz_min`/`mz_max` na tym etapie, żeby audyt duplikatów/jakości objął całą pulę, nie tylko to, co już pasuje do docelowego zakresu.

In [3]:
broad_filters = {
    "organism": "Mouse",
    "organism_part": "Ovary",
    "condition": ["Wildtype", "Wtype", "N/A"],
    "polarity": "Negative",
    "annotation_fdr": 0.1,
    "min_annotation_count": 1,
}
results = explorer.filter(broad_filters)
print(f"Found {len(results)} datasets")
display(results[["dataset_id", "name", "condition", "analyzer_type", "ionisation_source", "mz_min", "mz_max", "pixel_count"]])

METASPACE discovery:   0%|          | 0/3 [00:00<?, ?stage/s]

Current operation:   0%|          | 0/1 [00:00<?, ?operation/s]

Found 5 datasets


,dataset_id,name,condition,analyzer_type,ionisation_source,mz_min,mz_max,pixel_count
0,2025-02-17_08h53m39s,con2-root mean square,Wild type,FTICR,MALDI,92.908705,947.526476,8392
1,2025-02-17_08h54m37s,ko1-root mean square,Wild type,FTICR,MALDI,92.909170,947.526476,7309
2,2025-02-17_08h52m41s,con1-root mean square,Wild type,FTICR,MALDI,92.908705,947.526476,7042
3,2023-06-21_22h13m45s,control_day 5_slice 3,Wildtype,12T FTICR,MALDI,147.562166,1099.983703,1629
4,2023-06-21_22h13m29s,control_day 5_slice 2,Wildtype,12T FTICR,MALDI,147.906388,1099.967407,954


## 2. Ręczna kontrola jakości, której żadna reguła biblioteczna nie łapie

Ten sam skan co w `kidney_dataset_repaired.ipynb`: niedopasowanie gatunku/tkanki w nazwie, jawne oznaczenia testowe.

In [4]:
suspect_pattern = r"zebrafish|drosophila|\brat\b|\bhuman\b|\btest\b|\(test\)|calib|standard"
suspects = results[results["name"].str.contains(suspect_pattern, case=False, na=False, regex=True)]
display(suspects[["dataset_id", "name", "condition", "organisms", "pixel_count"]] if len(suspects) else "none found")

'none found'

**Dodatkowa, ovary-specyficzna uwaga (nie złapana przez powyższy skan):** dataset `ko1-root mean square` ma `condition="Wild type"`, ale prefiks `ko` w nazwie sugeruje "knockout" — model genetyczny, nie dziki typ. To za słaba przesłanka, żeby wykluczyć automatycznie (dwuliterowy prefiks to nie dowód), ale przy zaledwie 5 kandydatach w tej puli **każdy rekord ma duże znaczenie** — zdecydowanie zweryfikuj to ręcznie (np. w opisie projektu na METASPACE) przed użyciem tego datasetu jako wildtype.

## 3. Przegląd biblioteczny (`DatasetExplorer.review_current`)

Brak wbudowanego profilu `"ovary"` w `_PROFILES` (tylko `brain`/`liver`) — przekazuję `DatasetReviewProfile` z poziomu notebooka, tak jak w `kidney_dataset_repaired.ipynb`.

In [5]:
ovary_profile = DatasetReviewProfile(
    low_pixel_threshold=None,  # pula ma tylko 5 rekordów, minimum 954 pikseli -- przy tak małej próbie nie stawiam arbitralnego progu, każdy rekord i tak trafia do ręcznego przeglądu w całości.
    morphology_pattern=r"(?:follicle|corpus.?luteum|stroma|oocyte)",
    explicit_regional_names=frozenset(),
)
review = explorer.review_current(profile=ovary_profile)

print("available rules:", review.available_rules)
display(review.summary())

display(
    review.table.loc[
        review.table["duplicate_cluster_size"] > 1,
        ["duplicate_cluster_id", "dataset_id", "name", "pixel_count", "duplicate_confidence", "duplicate_excluded", "recommended_keeper_dataset_id"],
    ].sort_values(["duplicate_confidence", "duplicate_cluster_id"])
)
display(
    review.table.loc[
        review.table["mz_shift_qc_variant"] | review.table["morphology_hint"].eq("regional_or_microregion") | review.table["low_pixel_flag"],
        ["dataset_id", "name", "pixel_count", "mz_shift_qc_variant", "morphology_hint", "low_pixel_flag"],
    ]
)

available rules: ('high_confidence_duplicates', 'mz_shift_qc_variants', 'explicit_regional_fragments')


,rule,dataset_count
0,high_confidence_duplicates,0
1,mz_shift_qc_variants,0
2,explicit_regional_fragments,0


,duplicate_cluster_id,dataset_id,name,pixel_count,duplicate_confidence,duplicate_excluded,recommended_keeper_dataset_id


,dataset_id,name,pixel_count,mz_shift_qc_variant,morphology_hint,low_pixel_flag


## 4. Zastosowanie reguł i finalna lista

`high_confidence_duplicates` + `mz_shift_qc_variants` + `explicit_regional_fragments` (wszystkie trzy, dla spójności z pozostałymi notebookami, nawet gdy akurat wychodzi 0). `morphology_hint`/`low_pixel_flag` zostają doradczo.

In [6]:
applied_rules = ["high_confidence_duplicates", "mz_shift_qc_variants", "explicit_regional_fragments"]
explorer.apply_review(review, rules=applied_rules)

final_filters = {
    "organism": "Mouse",
    "organism_part": "Ovary",
    "condition": ["Wildtype", "Wtype", "N/A"],
    "polarity": "Negative",
    "mz_min": 200,
    "mz_max": 900,
    "annotation_fdr": 0.1,
    "min_annotation_count": 1,
    "include_molecule_stats": True,
    "include_spatial_annotation_stats": False,  # see brain_dataset.ipynb section 6 for the cost rationale
}
# exclude_dataset_ids intentionally omitted -- session-level exclusions from steps 2/3/4 persist across this re-query.
results_ovary = explorer.filter(final_filters)
print(f"final ovary shortlist: {len(results_ovary)} datasets (of {len(results)} broad candidates)")
display(results_ovary[["dataset_id", "name", "analyzer_type", "pixel_count", "molecule_count", "unique_molecule_count"]])

METASPACE discovery:   0%|          | 0/3 [00:00<?, ?stage/s]

Current operation:   0%|          | 0/1 [00:00<?, ?operation/s]

final ovary shortlist: 5 datasets (of 5 broad candidates)


,dataset_id,name,analyzer_type,pixel_count,molecule_count,unique_molecule_count
0,2025-02-17_08h53m39s,con2-root mean square,FTICR,8392,43,3
1,2025-02-17_08h54m37s,ko1-root mean square,FTICR,7309,49,3
2,2025-02-17_08h52m41s,con1-root mean square,FTICR,7042,43,1
3,2023-06-21_22h13m45s,control_day 5_slice 3,12T FTICR,1629,220,27
4,2023-06-21_22h13m29s,control_day 5_slice 2,12T FTICR,954,217,24


## 5. Grupowanie w serie biologiczne (Poziom 3)

Ta sama heurystyka co w `brain_dataset.ipynb` — tylko do identyfikacji grup, które muszą zostać razem w tym samym podziale train/validation/test, nie do automatycznego wybierania reprezentanta.

In [7]:
def biological_series_key(name: str) -> str:
    s = str(name).lower()
    s = re.sub(r"^\d{4}-\d{2}-\d{2}[_ ]", "", s)
    s = re.sub(r"^\d{8}_+", "", s)
    s = re.sub(r"_(?:aq_ml|aq|ml)$", "", s)
    s = re.sub(r"_\d+ppm$", "", s)
    s = re.sub(r"-total ion count$", "", s)
    s = re.sub(r" - root mean square$", "", s)
    s = re.sub(r"_replicate\d+$", "", s)
    s = re.sub(r"_s\d+$", "", s)
    s = re.sub(r"_\d+$", "", s)
    s = re.sub(r"[^a-z0-9]+", "_", s).strip("_")
    return s


results_ovary["biological_series_id"] = results_ovary["name"].apply(biological_series_key)
n_series = results_ovary["biological_series_id"].nunique()
print(f"final shortlist: {len(results_ovary)} datasets across {n_series} name-derived series")
series_sizes = results_ovary.groupby("biological_series_id").size().sort_values(ascending=False)
display(series_sizes[series_sizes > 1])

final shortlist: 5 datasets across 5 name-derived series


Series([], dtype: int64)

In [8]:
output_path = Path("data/ovary_workspace/configs/datasets/ovary")
exported = explorer.export_selection(output_path, sort_by="download_size_bytes", ascending=False)
exported

{'filters': PosixPath('data/ovary_workspace/configs/datasets/ovary/filter.json'),
 'selection': PosixPath('data/ovary_workspace/configs/datasets/ovary/selection.json')}

## Podsumowanie

- Szeroka pula (`Mouse`+`Negative`+`Wildtype`/`Wtype`/`N/A`): patrz sekcja 1 dla dokładnej liczby.
- Kontrola jakości i przegląd biblioteczny: sekcje 2–3.
- Zakres m/z `200–900` przyjęty jako domyślny, spójny z kidney — **nie zoptymalizowany osobno dla tego narządu**, patrz zastrzeżenie na początku notebooka.
- Eksport do `data/ovary_workspace/configs/datasets/ovary/` — pierwszy raz dla tego narządu, brak wcześniejszej konfiguracji do porównania.
- Analiza wspólnego zakresu m/z między wszystkimi narządami — świadomie pominięta, do rozwiązania osobno.